In [2]:
print("hello world")

hello world


In [3]:
import json
import shutil
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from silero_vad import get_speech_timestamps, load_silero_vad

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
chunks_root = root / "outputs" / "chunks"

MAX_SEC = 30  # split anything longer than this
MIN_SEC = 10  # no piece shorter than this (needs MIN_SEC <= MAX_SEC / 2)
SRC_FOLDER = "merged_speakers"  # the last chunk folder made in 04_diarization

# a chunk without speech is always deleted
MIN_SPEECH_SEC = 0.5  # less speech than this (Silero VAD) = no speech

# music / background filter, a chunk is rejected when ANY limit is passed (None = ignore that limit)
MIN_SNR_DB = 12           # loud parts vs quiet parts; low = music or noise under the voice
MAX_NOISE_DBFS = -25      # level of the quiet parts; high = constant background sound
MAX_OTHER_SOUND_PCT = 30  # % of the chunk that is loud (> -40 dBFS) but not speech (music, claps, ...)
DELETE_REJECTED = False   # False = move rejected wavs to <folder>/rejected/, True = delete them

vad_model = load_silero_vad()


def quietest_moment(x, sr, lo_sec, hi_sec, smooth_sec=0.2):
    """Time (s) in the middle of the quietest 200 ms between lo_sec and hi_sec."""
    a, b = int(lo_sec * sr), int(hi_sec * sr)
    win = max(int(smooth_sec * sr), 1)
    csum = np.cumsum(np.insert(x[a:b].astype(np.float64) ** 2, 0, 0.0))
    power = (csum[win:] - csum[:-win]) / win  # mean power in every 200 ms window
    return (a + int(np.argmin(power)) + win // 2) / sr


def silent_cuts(x, sr, max_sec=MAX_SEC, min_sec=MIN_SEC):
    """Cut times (s) that split x into pieces of min_sec..max_sec, each cut at a silent moment."""
    assert min_sec * 2 <= max_sec, "min_sec must be at most half of max_sec"
    total, cuts, start = len(x) / sr, [], 0.0
    while total - start > max_sec:
        lo = start + min_sec
        hi = min(start + max_sec, total - min_sec)  # leave at least min_sec for the rest
        start = quietest_moment(x, sr, lo, hi)
        cuts.append(start)
    return cuts


def split_long_chunks(chunks_dir, out_dir=None, max_sec=MAX_SEC, min_sec=MIN_SEC):
    """Copy every wav of chunks_dir (with its manifest.jsonl) to out_dir, cutting those over max_sec at silences."""
    chunks_dir = Path(chunks_dir)
    out_dir = Path(out_dir) if out_dir else chunks_dir.with_name(f"{chunks_dir.name}_{max_sec}s")
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob("*.wav"):  # leftovers from an earlier run
        old.unlink()

    rows = [json.loads(l) for l in (chunks_dir / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    out = []
    for row in rows:
        src = chunks_dir / Path(row["audio_filepath"]).name
        x, sr = sf.read(src, dtype="float32")
        idx = [0, *(round(c * sr) for c in silent_cuts(x, sr, max_sec, min_sec)), len(x)]
        for k, (a, b) in enumerate(zip(idx, idx[1:])):
            name = src.name if len(idx) == 2 else f"{src.stem}_{k:02d}.wav"
            sf.write(out_dir / name, x[a:b], sr)
            out.append({
                "audio_filepath": str(out_dir / name),
                "start": round(row["start"] + a / sr, 3),
                "end": round(row["start"] + b / sr, 3),
                "duration": round((b - a) / sr, 3),
                "speaker": row["speaker"],
                "source": src.name,
            })
    with open(out_dir / "manifest.jsonl", "w", encoding="utf-8") as f:
        for item in out:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    return out


def background_stats(x, sr):
    """Speech and music / background measurements of a chunk, or None if it is too short to measure."""
    if len(x) < 0.2 * sr:
        return None
    frame = int(0.05 * sr)  # 50 ms frames: loud parts = voice, quiet parts = background
    n = len(x) // frame
    rms = np.sqrt((x[: n * frame].reshape(n, frame).astype(np.float64) ** 2).mean(axis=1))
    dbfs = 20 * np.log10(np.maximum(rms, 1e-10))
    noise_dbfs = np.percentile(dbfs, 10)
    snr_db = np.percentile(dbfs, 90) - noise_dbfs

    stamps = get_speech_timestamps(torch.from_numpy(x), vad_model, sampling_rate=sr, threshold=0.5, speech_pad_ms=0)
    speech = np.zeros(len(x), dtype=bool)
    for t in stamps:
        speech[t["start"]:t["end"]] = True
    frame = int(0.1 * sr)  # 100 ms frames: loud but not speech = other sound
    n = len(x) // frame
    is_speech = speech[: n * frame].reshape(n, frame).mean(axis=1) >= 0.5
    loud = 20 * np.log10(np.maximum(np.sqrt((x[: n * frame].reshape(n, frame).astype(np.float64) ** 2).mean(axis=1)), 1e-10)) > -40
    return {
        "speech_sec": round(float(speech.sum() / sr), 2),
        "noise_dbfs": round(float(noise_dbfs), 1),
        "snr_db": round(float(snr_db), 1),
        "other_sound_pct": round(float((loud & ~is_speech).mean() * 100), 1),
    }


def noisy_reasons(s):
    """Which limits a chunk's background_stats pass."""
    reasons = []
    if MIN_SNR_DB is not None and s["snr_db"] < MIN_SNR_DB:
        reasons.append(f"snr {s['snr_db']} dB < {MIN_SNR_DB}")
    if MAX_NOISE_DBFS is not None and s["noise_dbfs"] > MAX_NOISE_DBFS:
        reasons.append(f"noise floor {s['noise_dbfs']} dBFS > {MAX_NOISE_DBFS}")
    if MAX_OTHER_SOUND_PCT is not None and s["other_sound_pct"] > MAX_OTHER_SOUND_PCT:
        reasons.append(f"other sound {s['other_sound_pct']}% > {MAX_OTHER_SOUND_PCT}")
    return reasons


def clean_noisy_chunks(out_dir, rows):
    """Delete chunks without speech, reject the ones with too much music / background.
    Rewrites manifest.jsonl, logs everything removed in rejected.jsonl."""
    out_dir = Path(out_dir)
    rejected_dir = out_dir / "rejected"
    for old in rejected_dir.glob("*.wav"):  # leftovers from an earlier run
        old.unlink()
    kept, rejected = [], []
    for row in rows:
        path = Path(row["audio_filepath"])
        x, sr = sf.read(path, dtype="float32")
        stats = background_stats(x, sr)
        if stats is None or stats["speech_sec"] < MIN_SPEECH_SEC:  # too short to hold speech, or no speech
            path.unlink()  # always deleted
            rejected.append({**row, **(stats or {}), "reasons": ["no speech"], "deleted": True})
            continue
        reasons = noisy_reasons(stats)
        if not reasons:
            kept.append({**row, **stats})
            continue
        if DELETE_REJECTED:
            path.unlink()
        else:
            rejected_dir.mkdir(exist_ok=True)
            row = {**row, "audio_filepath": str(shutil.move(path, rejected_dir / path.name))}
        rejected.append({**row, **stats, "reasons": reasons, "deleted": DELETE_REJECTED})
    for name, items in (("manifest.jsonl", kept), ("rejected.jsonl", rejected)):
        with open(out_dir / name, "w", encoding="utf-8") as f:
            for item in items:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
    return kept, rejected


# every audio that has the chunk folder: split long chunks, then drop the ones with music / background
for chunks_dir in sorted(chunks_root.glob(f"*/{SRC_FOLDER}")):
    if not (chunks_dir / "manifest.jsonl").exists():
        print("skip (no manifest):", chunks_dir.parent.name)
        continue
    pieces = split_long_chunks(chunks_dir)
    kept, rejected = clean_noisy_chunks(chunks_dir.with_name(f"{chunks_dir.name}_{MAX_SEC}s"), pieces)
    longest = max((p["duration"] for p in kept), default=0)
    no_speech = sum(r["reasons"] == ["no speech"] for r in rejected)
    print(f"{chunks_dir.parent.name[:60]:60} {len(pieces)} files -> kept {len(kept)}, "
          f"no speech {no_speech}, noisy {len(rejected) - no_speech}, longest {longest:.1f}s")

/Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak    2264 files -> kept 1518, no speech 581, noisy 165, longest 29.9s
2026-yilda Google'ga ishga kirish XARITASI                   91 files -> kept 90, no speech 0, noisy 1, longest 29.8s
28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avto 152 files -> kept 134, no speech 12, noisy 6, longest 29.5s
29-son - Krinjdan tortib siyosatgacha hafta davomida sodir b 138 files -> kept 124, no speech 11, noisy 3, longest 29.8s
31 - son - Birinçi besoqol, musulmon, tub amerikalik senator 168 files -> kept 153, no speech 7, noisy 8, longest 29.8s
40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maq 152 files -> kept 149, no speech 3, noisy 0, longest 29.5s
50 yoshida ham studentlikni tanlaydigan odam bilan suhbat    1216 files -> kept 813, no speech 235, noisy 168, longest 29.9s
99% odamlar buni eplay olmaydi - Iqtibos podcast #27         188 files -> kept 186, no speech 2, noisy 0, longest 29.9s
AQLI PASTlar avlodi (bo'lib q

In [6]:
# Clean the 30 s chunks with Mel-Band RoFormer:
#   1. vocals separation (Kim Vocal 2)  -> removes music / background sound
#   2. dereverb + de-echo (Sucial)      -> removes room reverb
#   3. mono, OUT_SR, 16-bit wav
# Needs: uv add "audio-separator[cpu]"   (Apple Silicon: PyTorch runs on MPS)
import json
import logging
import os
import re
import tempfile
from pathlib import Path

import soundfile as sf
import torch
import torchaudio.functional as AF
from audio_separator.separator import Separator
from tqdm.auto import tqdm

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
chunks_root = root / "outputs" / "chunks"

IN_FOLDER = "merged_speakers_30s"  # made by the cell above
OUT_FOLDER = "clean"  # same file names, the originals stay untouched
OUT_SR = 48000
USE_DEREVERB = True  # False = vocals separation only
LIMIT_PER_AUDIO = 5 # e.g. 5 = quick test on a few files per podcast, None = all
ONLY = None  # only podcast folders whose name contains this text, None = all

VOCALS_MODEL = "vocals_mel_band_roformer.ckpt"  # MelBand Roformer, Kimberley Jensen
DEREVERB_MODEL = "dereverb-echo_mel_band_roformer_sdr_13.4843_v2.ckpt"  # Sucial De-Reverb-Echo V2
# (v1, the one in the Hugging Face screenshot: "dereverb-echo_mel_band_roformer_sdr_10.0169.ckpt")
VOCALS_STEM = r"\(vocals\)"  # output files are named <input>_(<stem>)_<model>.wav
DEREVERB_STEM = r"\(no[ _-]?reverb\)"
MODELS_DIR = Path.home() / ".cache" / "audio-separator-models"
MODEL_SR = 44100  # the checkpoints work at 44.1 kHz, audio-separator resamples the input itself


def load_separator(model, out_dir):
    sep = Separator(output_dir=str(out_dir), output_format="WAV", model_file_dir=str(MODELS_DIR),
                    sample_rate=MODEL_SR, use_soundfile=True, log_level=logging.WARNING)
    sep.load_model(model_filename=model)
    return sep


def pick_stem(files, out_dir, pattern, what):
    """The one output file whose name matches pattern."""
    paths = [Path(out_dir) / Path(f).name for f in files]
    hits = [p for p in paths if re.search(pattern, p.name, re.I)]
    if len(hits) != 1:
        raise RuntimeError(f"{what}: expected one output matching /{pattern}/, got {[p.name for p in paths]} "
                           "-> fix VOCALS_STEM / DEREVERB_STEM")
    return hits[0]


def enhance(src, dst, first=False):
    """src wav -> vocals -> dereverb -> mono OUT_SR wav at dst. dst only appears when it is complete."""
    out = vocals.separate(str(src))
    x = pick_stem(out, tmp_vocals, VOCALS_STEM, "vocals stage")
    if first:
        print("vocals stage wrote:", out)
    if dereverb:
        out = dereverb.separate(str(x))
        x = pick_stem(out, tmp_dereverb, DEREVERB_STEM, "dereverb stage")
        if first:
            print("dereverb stage wrote:", out)
    audio, sr = sf.read(x, dtype="float32", always_2d=True)
    y = torch.from_numpy(audio.mean(axis=1))
    y = (AF.resample(y, sr, OUT_SR) if sr != OUT_SR else y).numpy().clip(-1.0, 1.0)
    dst.parent.mkdir(exist_ok=True)
    part = dst.with_name(dst.name + ".part")
    sf.write(part, y, OUT_SR, subtype="PCM_16", format="WAV")
    os.replace(part, dst)


# what to do: every kept chunk of every podcast that has no clean version yet
jobs, manifests = [], {}  # manifests: clean folder -> rows of its manifest.jsonl
for in_dir in sorted(chunks_root.glob(f"*/{IN_FOLDER}")):
    if ONLY and ONLY not in in_dir.parent.name:
        continue
    if not (in_dir / "manifest.jsonl").exists():
        print("skip (no manifest):", in_dir.parent.name)
        continue
    out_dir = in_dir.parent / OUT_FOLDER
    rows = [json.loads(l) for l in (in_dir / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    manifests[out_dir] = rows
    new = [(Path(r["audio_filepath"]), out_dir / Path(r["audio_filepath"]).name) for r in rows
           if not (out_dir / Path(r["audio_filepath"]).name).exists()]  # finished files are skipped -> safe to resume
    jobs += new[:LIMIT_PER_AUDIO] if LIMIT_PER_AUDIO else new
print(f"{sum(map(len, manifests.values()))} chunks in {len(manifests)} podcasts, {len(jobs)} still to clean")

failed = []
MODELS_DIR.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(prefix="enhance_") as tmp:
    tmp_vocals, tmp_dereverb = Path(tmp) / "vocals", Path(tmp) / "dereverb"
    tmp_vocals.mkdir(), tmp_dereverb.mkdir()
    vocals = load_separator(VOCALS_MODEL, tmp_vocals)
    dereverb = load_separator(DEREVERB_MODEL, tmp_dereverb) if USE_DEREVERB else None
    try:
        for n, (src, dst) in enumerate(tqdm(jobs, unit="file")):
            try:
                enhance(src, dst, first=n == 0)
            except Exception as e:  # one bad file must not stop a long run
                failed.append((src, f"{type(e).__name__}: {e}"))
                tqdm.write(f"FAILED {src.name}: {e}")
            finally:
                for d in (tmp_vocals, tmp_dereverb):
                    for f in d.glob("*"):
                        f.unlink(missing_ok=True)
    finally:  # also after Ctrl-C: the manifest lists exactly the files that exist
        for out_dir, rows in manifests.items():
            done = [{**r, "audio_filepath": str(out_dir / Path(r["audio_filepath"]).name), "sample_rate": OUT_SR}
                    for r in rows if (out_dir / Path(r["audio_filepath"]).name).exists()]
            if done:
                with open(out_dir / "manifest.jsonl", "w", encoding="utf-8") as f:
                    for item in done:
                        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"done, {len(failed)} failed")
for src, why in failed[:20]:
    print(" ", src.parent.parent.name[:40], src.name, why)

14983 chunks in 31 podcasts, 155 still to clean


  0%|          | 0/155 [00:00<?, ?file/s]2026-11-20 14:41:15,417 - WARNING - mdxc_separator - Audio duration (2.89s) is less than 10 seconds.
2026-11-20 14:41:15,417 - WARNING - mdxc_separator - Automatically enabling override_model_segment_size for better processing of short audio.
100%|██████████| 2/2 [00:00<00:00,  2.27it/s]
2026-11-20 14:41:16,304 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:41:16,308 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:41:16,396 - WARNING - mdxc_separator - Audio duration (2.89s) is less than 10 seconds.
2026-11-20 14:41:16,396 - WARNING - mdxc_separator - Automatically enabling override_model_segment_size for better processing of short audio.


vocals stage wrote: ['0000_speaker_0_(other)_vocals_mel_band_roformer.wav', '0000_speaker_0_(vocals)_vocals_mel_band_roformer.wav']


100%|██████████| 2/2 [00:00<00:00,  2.73it/s]
2026-11-20 14:41:17,137 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:41:17,141 - WARNING - common_separator - Using soundfile for writing.
  1%|          | 1/155 [00:01<04:36,  1.80s/file]2026-11-20 14:41:17,213 - WARNING - mdxc_separator - Audio duration (2.50s) is less than 10 seconds.
2026-11-20 14:41:17,213 - WARNING - mdxc_separator - Automatically enabling override_model_segment_size for better processing of short audio.


FAILED 0000_speaker_0.wav: dereverb stage: expected one output matching /\(no[ _-]?reverb\)/, got ['0000_speaker_0_(vocals)_vocals_mel_band_roformer_(No dry)_dereverb-echo_mel_band_roformer_sdr_13.wav', '0000_speaker_0_(vocals)_vocals_mel_band_roformer_(dry)_dereverb-echo_mel_band_roformer_sdr_13.wav'] -> fix VOCALS_STEM / DEREVERB_STEM


100%|██████████| 1/1 [00:00<00:00,  2.52it/s]
2026-11-20 14:41:17,616 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:41:17,618 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:41:17,677 - WARNING - mdxc_separator - Audio duration (2.50s) is less than 10 seconds.
2026-11-20 14:41:17,677 - WARNING - mdxc_separator - Automatically enabling override_model_segment_size for better processing of short audio.
100%|██████████| 1/1 [00:00<00:00,  2.69it/s]
2026-11-20 14:41:18,056 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:41:18,058 - WARNING - common_separator - Using soundfile for writing.
  1%|▏         | 2/155 [00:02<03:14,  1.27s/file]2026-11-20 14:41:18,121 - WARNING - mdxc_separator - Audio duration (2.02s) is less than 10 seconds.
2026-11-20 14:41:18,121 - WARNING - mdxc_separator - Automatically enabling override_model_segment_size for better processing of short audio.


... (53 outputs left out)

100%|██████████| 2/2 [00:03<00:00,  1.81s/it]
2026-11-20 14:45:26,567 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:45:26,575 - WARNING - common_separator - Using soundfile for writing.
100%|██████████| 5/5 [00:06<00:00,  1.28s/it]
2026-11-20 14:45:33,044 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:45:33,056 - WARNING - common_separator - Using soundfile for writing.
 19%|█▊        | 29/155 [04:17<24:37, 11.72s/file]

FAILED 0001_speaker_0_00.wav: dereverb stage: expected one output matching /\(no[ _-]?reverb\)/, got ['0001_speaker_0_00_(vocals)_vocals_mel_band_roformer_(No dry)_dereverb-echo_mel_band_roformer_sdr_13.wav', '0001_speaker_0_00_(vocals)_vocals_mel_band_roformer_(dry)_dereverb-echo_mel_band_roformer_sdr_13.wav'] -> fix VOCALS_STEM / DEREVERB_STEM


100%|██████████| 2/2 [00:03<00:00,  1.94s/it]
2026-11-20 14:45:37,040 - WARNING - common_separator - Using soundfile for writing.
2026-11-20 14:45:37,049 - WARNING - common_separator - Using soundfile for writing.
 19%|█▊        | 29/155 [04:29<19:31,  9.30s/file]


KeyboardInterrupt: 

In [5]:
print("done")

done
